In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

In [ ]:
train.head()

In [ ]:
train.dtypes.value_counts()

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

In [ ]:
garage_cols = ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond']

# Check if the NaN values are in the same rows
garage_nan = train[garage_cols].isnull()
print("Rows where ALL 5 columns in the Garage table are NaN:")
print((garage_nan.sum(axis=1) == 5).sum())

print("\nRows where ANY column in Garage is NaN:")
print((garage_nan.sum(axis=1) > 0).sum())

In [ ]:
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
bsmt_nan = train[bsmt_cols].isnull()

# Rows with partial NaN (1-4 missing, not all 5)
partial_nan_mask = (bsmt_nan.sum(axis=1) > 0) & (bsmt_nan.sum(axis=1) < 5)
problem_rows = train[partial_nan_mask]

# Show problematic rows + TotalBsmtSF to check if basement physically exists
print(problem_rows[bsmt_cols + ['TotalBsmtSF']])

### Basement columns — anomaly investigation

Unlike Garage (81 rows, all 5 cols NaN = no garage), Bsmt has:
- **37 rows**: all 5 cols NaN → no basement (impute 'None')
- **2 rows**: partial NaN → basement exists, data entry error

Strategy:
- Row 332 (`BsmtFinType2` missing, TotalBsmtSF=3206): impute with mode
- Row 948 (`BsmtExposure` missing, TotalBsmtSF=936): impute with mode
- Remaining 37 rows: impute 'None' (no basement)

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
missing_pct = (missing / len(train) * 100).round(1)

colors = ['#d62728' if pct > 80 else '#ff7f0e' if pct > 15 else '#2ca02c' 
          for pct in missing_pct]

missing_pct.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('% missing')
ax.set_title("Missing values by column")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/figures/01_missing_values.png', dpi=120, bbox_inches='tight')
plt.show()

## Missing values - what I found

Checked data_description.txt - for many columns, "NA" means "no feature" (no garage, no pool etc), not missing data

Verified with code:
- Garage: 81 rows have NaN in all 5 garage columns -> there is no garage (it's not NA)
- Bsmt: 37 rows same pattern -> no basement, but 2 rows have partial NaN (rows 332, 948) - basement exists, so it's real missing data

For preprocessing I'll split columns into:
- Cat "no feature" → fill with 'None'
- Num "no feature" → fill with 0
- Real missing → fill with median/mode

In [ ]:
print(f"Original skewness: {train['SalePrice'].skew()}")
print(f"After log1p skewness: {np.log1p(train['SalePrice']).skew()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train['SalePrice'], ax=axes[0])
sns.histplot(np.log1p(train['SalePrice']), ax=axes[1])

axes[0].set_title("Raw")
axes[0].set_xlabel("SalePrice")
axes[1].set_title("After log1p")
axes[1].set_xlabel("log1p(SalePrice)")
plt.tight_layout()
plt.savefig('../reports/figures/02_target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

### Target Distribution & Transformation

- The original `SalePrice` distribution is highly right-skewed (long tail), with skewness = **1.88**.
- After applying `log1p()`, the skewness is reduced to **0.12**, resulting in a distribution much closer to normal.

### Takeaway

- The model is trained on `log1p(SalePrice)` to better align with the **RMSLE** evaluation metric.
- Predictions are generated in log space and then converted back to the original price scale before final submission.

In [ ]:
train.select_dtypes(include=[np.number])

In [ ]:
pearson_corr = train.select_dtypes(include=[np.number]).corr(method='pearson')
top15 = pearson_corr['SalePrice'].abs().sort_values(ascending=False).drop('SalePrice').head(15)
top15

In [ ]:
selected = top15.index.tolist() + ['SalePrice']
corr_top15 = pearson_corr.loc[selected, selected]

plt.figure(figsize=(12, 10))
sns.heatmap(corr_top15, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title("Top 15 features correlated with SalePrice")
plt.tight_layout()
plt.savefig('../reports/figures/03_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

### Correlation (top 15 features)

**Top features (strongest correlation with SalePrice):**
- OverallQual: **0.79**
- GrLivArea: **0.71**
- GarageCars: **0.64**
- GarageArea: **0.62**
- TotalBsmtSF / 1stFlrSF: **0.61**

**Multicollinearity**
- GarageCars and GarageArea: **0.88**
- TotalBsmtSF and 1stFlrSF: **0.82**

For linear models multicollinearity can cause instability, so we may need to drop one feature from highly correlated pairs (over 0.8 collinearity)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(train['GrLivArea'], train['SalePrice'], alpha=0.4)
plt.xlabel('GrLivArea')
plt.ylabel('SalePrice')
plt.title('GrLivArea vs SalePrice')
plt.savefig('../reports/figures/04_outliers.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
outliers_mask = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
print(train[outliers_mask][['GrLivArea', 'SalePrice']])

In [ ]:
plt.scatter(train['GrLivArea'], train['SalePrice'], alpha=0.4, label='normal')
plt.scatter(train[outliers_mask]['GrLivArea'], 
            train[outliers_mask]['SalePrice'], 
            color='red', marker='x', label='outliers')
plt.legend()
plt.savefig('../reports/figures/04_outliers.png', dpi=120, bbox_inches='tight')
plt.show()

### Outliers

We can see 2 extreme outliers: houses with GrLivArea > 4000 but SalePrice < 200000

These observations are unusual, as very large houses should generally have high prices.

Decision: we will remove these outliers from the training data to improve model stability

In [ ]:
# Outliers identified - will be removed in preprocessing step (notebook 02)
print(f"Outliers to drop: {outliers_mask.sum()}")

## Summary

### Key findings
- **Missing values** 19 columns have NaN. Most are not really "missing", for columns 
  like GarageQual, GarageCond, GarageFinish NaN means "no feature exists" (verified on data_description.txt 
  and via code on Garage 81/81 and Bsmt 37/2).

- **Target distribution:** SalePrice is right-skewed (skew = 1.88). After `log1p` 
  transformation skew drops to 0.12, close to normal. Model will be trained in log-space 
  to align with Kaggle's RMSLE metric.

- **Top features:** Strongest correlations with SalePrice are OverallQual (0.79), GrLivArea (0.71), 
  and GarageCars (0.64)

- **Multicollinearity:** Pairs like GarageCars / GarageArea (corr 0.88) and TotalBsmtSF / 1stFlrSF (corr 0.82) 
  carry redundant information. Not really problematic for tree-based models, but can be a problem for linear models.

- **Outliers:** 2 houses with GrLivArea > 4000 and SalePrice < 300000. To be removed before training.


### Preprocessing plan

- Drop 2 outliers identified above.
- Fill missing values in 3 groups:
  1. "No feature" categorical columns (PoolQC, Alley, ...) -> fill with `'None'`
  2. "No feature" numeric (GarageYrBlt, MasVnrArea) -> fill with `0`
  3. Real missing (LotFrontage, Electrical) -> fill with median/mode
- Apply `np.log1p` on target SalePrice.
- One-hot encode categorical columns with `pd.get_dummies`.


### Model plan
- Compare 4 models: Decision Tree, Random Forest, XGBoost, LightGBM
- 5-fold cross-validation with RMSE on log-transformed target (= RMSLE)
- Hyperparameter tuning with Optuna on best model
- Final submission to Kaggle leaderboard
